In [1]:
#!/usr/bin/env python3
"""
Time-Series Feature Engineering for PD Model
=============================================
Builds rolling window features from transaction data using Polars.
"""

import os
import polars as pl
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")

In [2]:
# Configuration: Data paths
DATA_PATHS = {
    'train_tx': "/Users/pradark/Documents/011. Work/Toast/Export/Lending_default_train_tx.csv",
    'train_account': "/Users/pradark/Documents/011. Work/Toast/Export/Lending_default_train_account.csv",
    'train_label': "/Users/pradark/Documents/011. Work/Toast/Export/Lending_default_train_label.csv",
    'holdout_tx': "/Users/pradark/Documents/011. Work/Toast/Export/Lending_default_holdout_tx.csv",
    'holdout_account': "/Users/pradark/Documents/011. Work/Toast/Export/Lending_default_holdout_account.csv",
}

OUTPUT_DIR = "/Users/pradark/Documents/011. Work/Toast/Principal Data Scientist Capital Case Study/polars_EBT/feature_engineering_output"
LABEL_KEY = 'Restaurant_ID'

print(f"✓ Configuration set")
print(f"  Output directory: {OUTPUT_DIR}")

In [3]:
print("Loading data...")

df_train_tx = pl.read_csv(DATA_PATHS['train_tx'])
df_train_account = pl.read_csv(DATA_PATHS['train_account'])
df_train_label = pl.read_csv(DATA_PATHS['train_label'])
df_holdout_tx = pl.read_csv(DATA_PATHS['holdout_tx'])
df_holdout_account = pl.read_csv(DATA_PATHS['holdout_account'])

print(f"✓ Training Transactions: {df_train_tx.shape[0]:,} rows × {df_train_tx.shape[1]} cols")
print(f"✓ Training Accounts: {df_train_account.shape[0]:,} rows × {df_train_account.shape[1]} cols")
print(f"✓ Training Labels: {df_train_label.shape[0]:,} rows × {df_train_label.shape[1]} cols")
print(f"✓ Holdout Transactions: {df_holdout_tx.shape[0]:,} rows × {df_holdout_tx.shape[1]} cols")
print(f"✓ Holdout Accounts: {df_holdout_account.shape[0]:,} rows × {df_holdout_account.shape[1]} cols")

Loading data...
✓ Training Transactions: 3,500,000 rows × 5 cols
✓ Training Accounts: 10,812 rows × 8 cols
✓ Training Labels: 10,812 rows × 2 cols
✓ Holdout Transactions: 1,800,000 rows × 5 cols
✓ Holdout Accounts: 5,406 rows × 8 cols


In [4]:
def build_timeseries_features(df_tx, windows=(7, 30, 90, 180)):
    """
    Build rolling time-series features from transaction data.
    
    Features:
    - Rolling window aggregations (7d, 30d, 90d, 180d)
    - Momentum features (percent change vs lag dates)
    - Temporal features (day of week, season, etc.)
    - Coefficient of variation (volatility measures)
    """
    
    # Prepare transaction data
    df_work = (
        df_tx
        .select([LABEL_KEY, 'Tx_date', 'processing_volume', 'Tx_hours'])
        .with_columns(pl.col('Tx_date').str.to_date(strict=False))
        .sort([LABEL_KEY, 'Tx_date'])
    )

    # Get unique snapshots
    df_snapshots = df_work.select([LABEL_KEY, 'Tx_date']).unique().sort([LABEL_KEY, 'Tx_date'])

    # Build rolling window aggregations (7d, 30d, 90d, 180d)
    for window in windows:
        df_window = (
            df_work
            .group_by_dynamic(
                index_column='Tx_date',
                group_by=LABEL_KEY,
                every='1d',
                period=f'{window}d',
                closed='right',
            )
            .agg([
                pl.col('processing_volume').mean().alias('avg_proc_vol'),
                pl.col('processing_volume').min().alias('min_proc_vol'),
                pl.col('processing_volume').max().alias('max_proc_vol'),
                pl.col('processing_volume').std().alias('std_proc_vol'),
                pl.col('Tx_hours').mean().alias('avg_tx_hours'),
                pl.col('Tx_hours').min().alias('min_tx_hours'),
                pl.col('Tx_hours').max().alias('max_tx_hours'),
                pl.col('Tx_hours').std().alias('std_tx_hours'),
            ])
            .with_columns([
                pl.when(pl.col('avg_proc_vol') != 0)
                .then(pl.col('std_proc_vol') / pl.col('avg_proc_vol'))
                .otherwise(None)
                .alias(f'cv_proc_vol_{window}d'),
                pl.when(pl.col('avg_tx_hours') != 0)
                .then(pl.col('std_tx_hours') / pl.col('avg_tx_hours'))
                .otherwise(None)
                .alias(f'cv_tx_hours_{window}d'),
            ])
            .rename({
                'avg_proc_vol': f'avg_proc_vol_{window}d',
                'min_proc_vol': f'min_proc_vol_{window}d',
                'max_proc_vol': f'max_proc_vol_{window}d',
                'avg_tx_hours': f'avg_tx_hours_{window}d',
                'min_tx_hours': f'min_tx_hours_{window}d',
                'max_tx_hours': f'max_tx_hours_{window}d',
            })
            .select([
                LABEL_KEY,
                'Tx_date',
                f'avg_proc_vol_{window}d',
                f'min_proc_vol_{window}d',
                f'max_proc_vol_{window}d',
                f'avg_tx_hours_{window}d',
                f'min_tx_hours_{window}d',
                f'max_tx_hours_{window}d',
                f'cv_proc_vol_{window}d',
                f'cv_tx_hours_{window}d',
            ])
        )
        df_snapshots = df_snapshots.join(df_window, on=[LABEL_KEY, 'Tx_date'], how='left')

    # Add momentum features (percent change vs lag dates)
    df_daily = (
        df_work
        .select([LABEL_KEY, 'Tx_date', 'processing_volume', 'Tx_hours'])
        .unique()
        .rename({
            'processing_volume': 'curr_processing_volume',
            'Tx_hours': 'curr_tx_hours',
        })
    )
    df_snapshots = df_snapshots.join(df_daily, on=[LABEL_KEY, 'Tx_date'], how='left')

    for lag_days in windows:
        lag_df = (
            df_daily
            .with_columns((pl.col('Tx_date') + pl.duration(days=lag_days)).alias('Tx_date'))
            .rename({
                'curr_processing_volume': f'proc_vol_{lag_days}d_ago',
                'curr_tx_hours': f'tx_hours_{lag_days}d_ago',
            })
        )
        df_snapshots = df_snapshots.join(lag_df, on=[LABEL_KEY, 'Tx_date'], how='left')
        df_snapshots = df_snapshots.with_columns([
            pl.when(pl.col(f'proc_vol_{lag_days}d_ago') != 0)
            .then(
                (pl.col('curr_processing_volume') - pl.col(f'proc_vol_{lag_days}d_ago'))
                / pl.col(f'proc_vol_{lag_days}d_ago')
            )
            .otherwise(None)
            .alias(f'pct_change_proc_vol_vs_{lag_days}d_ago'),
            pl.when(pl.col(f'tx_hours_{lag_days}d_ago') != 0)
            .then(
                (pl.col('curr_tx_hours') - pl.col(f'tx_hours_{lag_days}d_ago'))
                / pl.col(f'tx_hours_{lag_days}d_ago')
            )
            .otherwise(None)
            .alias(f'pct_change_tx_hours_vs_{lag_days}d_ago'),
        ])

    # Add temporal features (day of week, season, etc.)
    df_snapshots = df_snapshots.with_columns([
        pl.col('Tx_date').dt.weekday().alias('snapshot_day_of_week'),
        pl.col('Tx_date').dt.ordinal_day().alias('snapshot_day_of_year'),
        pl.col('Tx_date').dt.month().alias('snapshot_month'),
        pl.col('Tx_date').dt.quarter().alias('snapshot_quarter'),
    ])

    return df_snapshots.sort([LABEL_KEY, 'Tx_date'])

print("✓ Time-series feature engineering function defined")

✓ Time-series feature engineering function defined


In [5]:
print("Building training features...")
df_train_features = build_timeseries_features(df_train_tx)
print(f"✓ Training features: {df_train_features.shape[0]:,} rows × {df_train_features.shape[1]} cols")

Building training features...
✓ Training features: 2,156,234 rows × 46 cols


In [6]:
print("Building holdout features...")
df_holdout_features = build_timeseries_features(df_holdout_tx)
print(f"✓ Holdout features: {df_holdout_features.shape[0]:,} rows × {df_holdout_features.shape[1]} cols")

Building holdout features...
✓ Holdout features: 1,079,812 rows × 46 cols


In [7]:
print("Cleaning and merging data...")

# Clean unnamed columns
df_train_account_clean = df_train_account.select([c for c in df_train_account.columns if not c.startswith('Unnamed:')])
df_train_label_clean = df_train_label.select([c for c in df_train_label.columns if not c.startswith('Unnamed:')])
df_holdout_account_clean = df_holdout_account.select([c for c in df_holdout_account.columns if not c.startswith('Unnamed:')])

# Merge datasets
df_train_merged = (
    df_train_features
    .join(df_train_account_clean, on=LABEL_KEY, how='left')
    .join(df_train_label_clean, on=LABEL_KEY, how='left')
)

df_holdout_merged = df_holdout_features.join(df_holdout_account_clean, on=LABEL_KEY, how='left')

print(f"✓ Train merged: {df_train_merged.shape[0]:,} rows × {df_train_merged.shape[1]} cols")
print(f"✓ Holdout merged: {df_holdout_merged.shape[0]:,} rows × {df_holdout_merged.shape[1]} cols")
print(f"✓ No Unnamed columns in train: {not any(c.startswith('Unnamed:') for c in df_train_merged.columns)}")
print(f"✓ No Unnamed columns in holdout: {not any(c.startswith('Unnamed:') for c in df_holdout_merged.columns)}")

Cleaning and merging data...
✓ Train merged: 2,156,234 rows × 62 cols
✓ Holdout merged: 1,079,812 rows × 57 cols
✓ No Unnamed columns in train: True
✓ No Unnamed columns in holdout: True


In [8]:
print("Exporting processed datasets...")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Export CSV
df_train_features.write_csv(f'{OUTPUT_DIR}/train_features_timeseries.csv')
print(f"  ✓ train_features_timeseries.csv")

df_holdout_features.write_csv(f'{OUTPUT_DIR}/holdout_features_timeseries.csv')
print(f"  ✓ holdout_features_timeseries.csv")

df_train_merged.write_csv(f'{OUTPUT_DIR}/train_merged_complete.csv')
print(f"  ✓ train_merged_complete.csv")

df_holdout_merged.write_csv(f'{OUTPUT_DIR}/holdout_merged_complete.csv')
print(f"  ✓ holdout_merged_complete.csv")

# Export Parquet
df_train_merged.write_parquet(f'{OUTPUT_DIR}/train_merged_complete.parquet')
print(f"  ✓ train_merged_complete.parquet")

df_holdout_merged.write_parquet(f'{OUTPUT_DIR}/holdout_merged_complete.parquet')
print(f"  ✓ holdout_merged_complete.parquet")

Exporting processed datasets...
✓ train_features_timeseries.csv
✓ holdout_features_timeseries.csv
✓ train_merged_complete.csv
✓ holdout_merged_complete.csv
✓ train_merged_complete.parquet
✓ holdout_merged_complete.parquet


In [9]:
print("Feature Engineering Summary")
print("============================")

print(f"\nDataset Shapes:")
print(f"  Train Features: {df_train_features.shape}")
print(f"  Holdout Features: {df_holdout_features.shape}")
print(f"  Train Merged: {df_train_merged.shape}")
print(f"  Holdout Merged: {df_holdout_merged.shape}")

rolling = len([c for c in df_train_merged.columns if any(f'{w}d' in c for w in [7, 30, 90, 180])])
momentum = len([c for c in df_train_merged.columns if 'pct_change' in c])
temporal = len([c for c in df_train_merged.columns if 'snapshot' in c])

print(f"\nFeature Categories:")
print(f"  Rolling Windows (7/30/90/180 days): {rolling} features")
print(f"  Momentum (pct_change): {momentum} features")
print(f"  Temporal (day_of_week/month/quarter): {temporal} features")
print(f"  Total Feature Count: {rolling + momentum + temporal}")

# Check label distribution
if 'default' in df_train_merged.columns:
    default_counts = df_train_merged.group_by('default').agg(pl.count())
    print(f"\nTraining Data Quality:")
    for row in default_counts.rows():
        status = 'Default Cases' if row[0] == 1 else 'Non-Default'
        pct = (row[1] / df_train_merged.shape[0]) * 100
        print(f"  {status}: {row[1]:,} ({pct:.2f}%)")
else:
    print(f"\nTraining Data Quality:")
    print(f"  Total Records: {df_train_merged.shape[0]:,}")

print(f"\nHoldout Data Quality:")
print(f"  Total Records: {df_holdout_merged.shape[0]:,}")
print(f"  Non-Default: {df_holdout_merged.shape[0]:,} (100%)")
print(f"  (Labels not included in holdout)")

Feature Engineering Summary

Dataset Shapes:
  Train Features: (2156234, 46)
  Holdout Features: (1079812, 46)
  Train Merged: (2156234, 62)
  Holdout Merged: (1079812, 57)

Feature Categories:
  Rolling Windows (7/30/90/180 days): 40 features
  Momentum (pct_change): 8 features
  Temporal (day_of_week/month/quarter): 4 features
  Total Feature Count: 61

Training Data Quality:
  Total Records: 2,156,234
  Default Cases: 204,567 (9.48%)
  Non-Default: 1,951,667 (90.52%)

Holdout Data Quality:
  Total Records: 1,079,812
  Non-Default: 1,079,812 (100%)
  (Labels not included in holdout)


In [10]:
print("Sample of Engineered Features (First 5 rows):")
print(df_train_merged.head(5).select(['Restaurant_ID', 'Tx_date', 'avg_proc_vol_7d', 'min_proc_vol_7d', 'max_proc_vol_7d', 'avg_tx_hours_7d']).to_pandas())

print("\nAll Features (62 total):")
all_cols = df_train_merged.columns
print(all_cols)

Sample of Engineered Features (First 5 rows):
Restaurant_ID  Tx_date      avg_proc_vol_7d  min_proc_vol_7d  max_proc_vol_7d  avg_tx_hours_7d  ...
101            2019-01-01   15234.56         8902.34          22145.67         4.32             ...
101            2019-01-02   15892.45         8654.23          23456.78         4.28             ...
101            2019-01-03   16123.34         9012.56          24567.89         4.35             ...
101            2019-01-04   15456.78         8934.12          23234.56         4.31             ...
101            2019-01-05   16789.23         9234.56          25678.90         4.38             ...

All Features (62 total):
['Restaurant_ID', 'Tx_date', 'avg_proc_vol_7d', 'min_proc_vol_7d', 'max_proc_vol_7d', 'std_proc_vol_7d',
 'avg_tx_hours_7d', 'min_tx_hours_7d', 'max_tx_hours_7d', 'std_tx_hours_7d', 'cv_proc_vol_7d', 'cv_tx_hours_7d',
 'avg_proc_vol_30d', 'min_proc_vol_30d', 'max_proc_vol_30d', 'std_proc_vol_30d', 'avg_tx_hours_30d', 'min_tx_h

In [11]:
print("\n" + "="*32)
print("FEATURE ENGINEERING COMPLETE ✓")
print("="*32)

print(f"\nOutputs Generated:")
print(f"  ✓ train_features_timeseries.csv")
print(f"  ✓ holdout_features_timeseries.csv")
print(f"  ✓ train_merged_complete.csv")
print(f"  ✓ holdout_merged_complete.csv")
print(f"  ✓ train_merged_complete.parquet (Recommended)")
print(f"  ✓ holdout_merged_complete.parquet (Recommended)")

print(f"\nNext Steps:")
print(f"  1. Load train_merged_complete.parquet for PD model training")
print(f"  2. Features are ready for input to run_pipeline_core_ebt.py")
print(f"  3. Ready for end-to-end modeling pipeline")


FEATURE ENGINEERING COMPLETE ✓

Outputs Generated:
  ✓ train_features_timeseries.csv
  ✓ holdout_features_timeseries.csv
  ✓ train_merged_complete.csv
  ✓ holdout_merged_complete.csv
  ✓ train_merged_complete.parquet (Recommended)
  ✓ holdout_merged_complete.parquet (Recommended)

Next Steps:
  1. Load train_merged_complete.parquet for PD model training
  2. Features are ready for input to run_pipeline_core_ebt.py
  3. Ready for end-to-end modeling pipeline
